# Prototype

In [1]:
import sys
import pandas as pd
import duckdb
from duckdb import IOException
import altair as alt

import datetime as dt

In [2]:
SQL_CLEAN_CUSTOMERS = """
CREATE TABLE IF NOT EXISTS customers_clean AS (
SELECT
    customer_id::VARCHAR AS customer_id,
    try_cast(signup_date AS DATE) AS signup_date,  -- this takes out our bad dates, they will not be used in the following calculation
    upper(country)::VARCHAR AS country
FROM customers_raw);
"""
SQL_CLEAN_SUBSCRIPTIONS = """
CREATE TABLE IF NOT EXISTS subscriptions_clean AS (
SELECT
    customer_id::VARCHAR as customer_id,
    try_cast(start_date AS DATE) as start_date,
    try_cast(end_date AS DATE) as end_date,
    upper(plan) as plan,
    try_cast(monthly_price AS INTEGER) as monthly_price
FROM subscriptions_raw);
"""


def load_data(conn, customers_csv: str, subscriptions_csv: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load data CSVs and cast to proper data types
    """
    # Read raw data from CSVs
    try:
        conn.execute("""CREATE TABLE customers_raw AS SELECT * FROM read_csv(?);""", [customers_csv]).df()
    except IOException:
        print(f"File {customers_csv} does not exist")
        sys.exit()
    try:
        conn.execute("""CREATE TABLE subscriptions_raw AS SELECT * FROM read_csv(?);""", [subscriptions_csv]).df()
    except IOException:
        print(f"File {subscriptions_csv} does not exist")
        sys.exit()

    # Clean up data types
    conn.execute(SQL_CLEAN_CUSTOMERS)
    conn.execute(SQL_CLEAN_SUBSCRIPTIONS)

    # generate our dataframes
    customers = conn.execute("SELECT * FROM customers_clean").df()
    subscriptions = conn.execute("SELECT * FROM subscriptions_clean").df()


conn = duckdb.connect(":memory:")
load_data(conn, '../data/customers.csv', '../data/subscriptions.csv')


In [3]:
def get_mmr(conn):
    """
    Calculate monthly recurring revenue.
    """
    # Assumptions
    # - a partial month is counted as a full month for income purposes
    # - subscriptions without an end date is still active at the present
    # - one customer can have several active subscriptions
    # - if subscriber is not in subscribers.csv we filter it out (see below comments in SQL)
    # - if text is present in monthly price in our CSVs we filter it out (see below comments in SQL)

    SQL = """
    -- extend still open subscriptions to today, create a list of months
    with active_months AS (
        SELECT
            subs.customer_id,
            COALESCE(monthly_price, 0) as monthly_price, -- make query safe from null in monthly_price column (there is "thirty" in one cell, we treat that as 0)
            plan,
            generate_series(
                date_trunc('month', start_date),
                date_trunc('month', COALESCE(end_date, today()::DATE)),
                '1 month'::INTERVAL
            ) as months
        FROM
            subscriptions_clean as subs
        INNER JOIN
            customers_clean as custs
        ON
            subs.customer_id = custs.customer_id  -- here we get rid of non-existent subscribers (see C050 and C999 in the excercise data)
        GROUP BY
            ALL
    ),
    -- user the list of months to create a row per active month
    monthly_entries AS (
        SELECT
            customer_id,
            monthly_price,
            plan,
            unnest(months) as month -- start date of an active month
        FROM
            active_months)
    -- calculate 
    SELECT
        month,
        sum(monthly_price) as mmr
    FROM
        monthly_entries
    GROUP BY
        month
    ORDER BY
        month;
    """

    results = conn.sql(SQL).df() # get results as pandas DataFrame
    return alt.Chart(results).mark_line(interpolate="step-after").encode(x="month", y=alt.X("mmr").title("Income (euros)")).properties(width=800, title="MMR")  # Plot MMR over time


get_mmr(conn)

alt.Chart(...)

In [107]:
import datetime



def get_churn(conn):
    SQL = """
    -- select entries where there is an end date, sort them by customer and end date
    CREATE OR REPLACE TABLE churn_json AS (
        WITH lagged AS (
            SELECT
                customer_id,
                start_date,
                end_date,
                plan,
                monthly_price,
    
                -- If current entry has an end date we check:
                -- * if the current customer and the next are the same
                -- * if a new subscription was opened quickly enough
                -- then no churn else churn.
                -- If current entry has no end date, also no churn.
                CASE
                    WHEN customer_id = lead(customer_id) OVER()
                    THEN lead(start_date) OVER() - end_date
                    ELSE null
                END AS delta_t,
                customer_id = lead(customer_id) OVER() AS has_next,
                end_date IS NULL as no_end_date,
    
                -- collapsing the above into one boolean:
                NOT ((has_next AND delta_t < 30) OR end_date IS NULL) AS is_churn
                
            FROM subscriptions_clean
            ORDER BY customer_id, start_date
        )
        SELECT
            customer_id,
            start_date,
            end_date,
            plan,
            monthly_price,
            is_churn
        FROM lagged
        WHERE is_churn = true
    );
    """
    conn.execute(SQL)


get_churn(conn)

conn.execute("SELECT * FROM churn_json;").df()
conn.execute("COPY (SELECT * FROM churn_json) TO 'churn.json';")